In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


In [2]:
!pip install dagshub mlflow -q

import dagshub
import mlflow

dagshub.init(repo_owner='lshek22', repo_name='IEEE-CIS-Fraud-Detection', mlflow=True)
mlflow.set_experiment("Random_Logistic_Regression")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 88.9 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 73.3 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 40.1 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=1f4670ce-fcc6-42b9-a6ce-c9f9b9151480&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=3247c7e729e74da93b105849f36099075a0802236a57c2e0ca22b3bcde76eda0




Accessing as lshek22

Initialized MLflow to track repo "lshek22/IEEE-CIS-Fraud-Detection"

Repository lshek22/IEEE-CIS-Fraud-Detection initialized!

<Experiment: artifact_location='mlflow-artifacts:/0ec3efaa011a4fa6830997175190d778', creation_time=1777903602645, experiment_id='2', last_update_time=1777903602645, lifecycle_stage='active', name='Random_Logistic_Regression', tags={'mlflow.experimentKind': 'custom_model_development'}, trace_location=None, workspace='default'>

In [3]:
train = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv').merge(
        pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv'), on='TransactionID', how='left')
test = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv').merge(
       pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv'), on='TransactionID', how='left')


In [4]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.metrics import roc_auc_score
import mlflow

X = train.drop(columns=['isFraud'])
y = train['isFraud']

cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
num_cols = X.select_dtypes(exclude=['object', 'category']).columns.tolist()

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

preprocessor_linear = ColumnTransformer([
    ('num', Pipeline([
        ('imp', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]), num_cols),
    ('cat', Pipeline([
        ('imp', SimpleImputer(strategy='most_frequent')),
        ('enc', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)),
        ('scaler', StandardScaler()) 
    ]), cat_cols)
])

In [6]:
def log_run_linear(run_name, model, params, registered_model_name=None):
    with mlflow.start_run(run_name=run_name):
        p = Pipeline([('pre', preprocessor_linear), ('model', model)])
        p.fit(X_train, y_train)

        tr_auc = roc_auc_score(y_train, p.predict_proba(X_train)[:,1])
        vl_auc = roc_auc_score(y_val,   p.predict_proba(X_val)[:,1])

        mlflow.log_params(params)
        mlflow.log_metrics({'train_auc': tr_auc, 'val_auc': vl_auc})

        mlflow.sklearn.log_model(
            sk_model=p,
            name="model",
            registered_model_name=registered_model_name
        )
        print(f'{run_name:45s} train={tr_auc:.4f} val={vl_auc:.4f}')
        return p, vl_auc

In [7]:
p_lr1, a_lr1 = log_run_linear('LR_C0001_underfit',
    LogisticRegression(C=0.0001, max_iter=1000, class_weight='balanced', n_jobs=-1),
    {'C': 0.0001, 'penalty': 'l2'},
            registered_model_name='LR_C0001_underfit')

p_lr2, a_lr2 = log_run_linear('LR_C1_balanced',
    LogisticRegression(C=1.0, max_iter=1000, class_weight='balanced', n_jobs=-1),
    {'C': 1.0, 'penalty': 'l2'},
                              registered_model_name='LR_C1_balanced')

candidates_lr = [(p_lr1, a_lr1, 'LR_C0001'), (p_lr2, a_lr2, 'LR_C1')]
best_lr = max(candidates_lr, key=lambda x: x[1])
best_lr_pipeline, best_lr_auc, best_lr_name = best_lr
print(f'Best LR: {best_lr_name} — val_auc={best_lr_auc:.4f}')

with mlflow.start_run(run_name='LR_Best_Final'):
    mlflow.log_metrics({'val_auc': best_lr_auc})
    mlflow.sklearn.log_model(
        sk_model=best_lr_pipeline,
        name="model",
        registered_model_name="IEEEFraud_LR"
    )
print('Registered.')

2026/05/05 17:15:27 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'LR_C0001_underfit'.
2026/05/05 17:15:43 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: LR_C0001_underfit, version 1
Created version '1' of model 'LR_C0001_underfit'.


LR_C0001_underfit                             train=0.8589 val=0.8569
🏃 View run LR_C0001_underfit at: https://dagshub.com/lshek22/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/2/runs/a99afbd699554405ba7abb1e25155e0c
🧪 View experiment at: https://dagshub.com/lshek22/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/2


2026/05/05 17:19:03 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'LR_C1_balanced'.
2026/05/05 17:19:11 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: LR_C1_balanced, version 1
Created version '1' of model 'LR_C1_balanced'.


LR_C1_balanced                                train=0.8696 val=0.8634
🏃 View run LR_C1_balanced at: https://dagshub.com/lshek22/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/2/runs/b423e4c92ba6416093c23686ce6621e4
🧪 View experiment at: https://dagshub.com/lshek22/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/2
Best LR: LR_C1 — val_auc=0.8634


2026/05/05 17:19:22 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'IEEEFraud_LR'.
2026/05/05 17:19:31 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: IEEEFraud_LR, version 1
Created version '1' of model 'IEEEFraud_LR'.


🏃 View run LR_Best_Final at: https://dagshub.com/lshek22/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/2/runs/1a92aaf94c2e4e11a6c6ee5931e019f8
🧪 View experiment at: https://dagshub.com/lshek22/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/2
Registered.
